# gVCF to VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1744690683255_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-108-59.ap-southeast-1.compute.internal:39247
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /mnt/yarn/usercache/livy/appcache/application_1744690683255_0001/container_1744690683255_0001_01_000001/hail-20250415-0432-0.2.134-952ae203dbbe.log

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [33]:
gvcf_list_path='s3://...'
vds_prefix = 's3://...'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [34]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

10322
2025-04-15 05:09:22.401 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [35]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [41]:
ls_gvcf[0:100]

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

['s3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6375/6de5907f-6049-4852-99c7-adfda64d6889/output/try-1/WHB6375.hard-filtered.gvcf.gz', 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6377/64a24a8b-25fd-42a8-93f2-22ebfd581706/output/try-1/WHB6377.hard-filtered.gvcf.gz', 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6378/e272cfc6-81ea-49b4-8694-4413c7907e9f/output/try-1/WHB6378.hard-filtered.gvcf.gz', 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6380/c5169919-0140-411e-bf2b-765553f35f1f/output/try-1/WHB6380.hard-filtered.gvcf.gz', 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6381/f2a39ab6-0342-4980-8b31-791a41f4e549/output/try-1/WHB6381.hard-filtered.gvcf.gz', 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6300/fdd1b13b-7452-45c0-954a-80560830e731/output/try-1/WHB6300.hard-filtered.gvcf.gz', 's3://precise-wgs-databundle/byob/sg10k-wgs-dragen/WHB6385/2af836d8-43b0-442f-ba4d-6f463f24c74e/output/try-1/WHB6385.hard-filtered.gvcf.gz', 's3://precis

In [ ]:
# Combine gVCF
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_100.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[0:100],
    use_genome_default_intervals=True,
    reference_genome='GRCh38' 
)

combiner.run()